In [ ]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)
stores = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
categories = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    tx_num = random.randint(1, 9999)
    return {
        'tx_id': f'TX{tx_num:04d}',
        'user_id': f'u{random.randint(1, 20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(stores),
        'category': random.choice(categories),
        'timestamp': datetime.now().isoformat(),
}
for i in range(50):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1:02d}] {tx['tx_id']} | {tx['amount']:8.2f} PLN | {tx['store']}")
    time.sleep(1)
   
print("Gotowe.")

In [ ]:
%%file consumer_anomaly.py
from kafka import KafkaConsumer
import json
from datetime import datetime
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='anomaly_detector_group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na anomalie prędkości...")
user_transaction_times = {}
for message in consumer:
    transaction = message.value
    
    user_id = transaction.get('user_id')
    tx_time_str = transaction.get('timestamp')
        
    if user_id not in user_transaction_times:
        user_transaction_times[user_id] = []
        
    user_transaction_times[user_id].append(tx_time)
    
    active_window_transactions = [
        t for t in user_transaction_times[user_id] 
        if (tx_time - t).total_seconds() <= 60
    ]
    user_transaction_times[user_id] = active_window_transactions
    
    if len(active_window_transactions) > 3:
        print(f"ALERT: Wykryto anomalię prędkości dla użytkownika {user_id}! "
              f"Liczba transakcji w ciągu ostatnich 60s: {len(active_window_transactions)}. ")